# 개별종목 조합F — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합F 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합F의 피처 값만 지정합니다.
import json

COMBINATION = 'F'
FEATURE_COLUMNS = (
    'ret_5_rank',
    'sector_relative_rank',
    'turnover_rank',
    'hv_20_rank',
    'market_cap_percentile',
    'sector_market_cap_rank',
    'industry_stock_rank',
    'volume_z_20',
    'bb_position',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합F 피처: ('ret_5_rank', 'sector_relative_rank', 'turnover_rank', 'hv_20_rank', 'market_cap_percentile', 'sector_market_cap_rank', 'industry_stock_rank', 'volume_z_20', 'bb_position')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.4515,0.5012,-0.0497,0.3908,0.3927,0.0978,0.3899,0.3163,0.3780
1,2,balanced,980,20150123,20150421,0.3885,0.3978,-0.0093,0.3732,0.3738,0.0672,0.3645,0.2820,0.3409
2,3,balanced,1210,20151228,20160328,0.3539,0.3762,-0.0223,0.3430,0.3452,0.0224,0.3493,0.2704,0.3178
3,4,balanced,1439,20161202,20170228,0.4178,0.4617,-0.0439,0.3942,0.3942,0.1004,0.4110,0.3224,0.3735
4,5,balanced,1669,20171113,20180207,0.3879,0.3901,-0.0022,0.3732,0.3754,0.0676,0.3800,0.3014,0.3498
5,6,balanced,1899,20181024,20190118,0.4060,0.3725,0.0335,0.3932,0.3957,0.0987,0.4096,0.2753,0.3473
6,7,balanced,2129,20190930,20191224,0.4389,0.4781,-0.0393,0.4128,0.4141,0.1289,0.4122,0.3396,0.3924
7,8,balanced,2359,20200902,20201130,0.3854,0.3476,0.0378,0.3793,0.3852,0.0773,0.3871,0.3490,0.3705
8,9,balanced,2589,20210806,20211105,0.3866,0.3914,-0.0048,0.3737,0.3843,0.0715,0.3849,0.2770,0.3381
9,10,balanced,2818,20220714,20221012,0.3395,0.3454,-0.0059,0.3281,0.3368,0.0078,0.3599,0.2475,0.2990


,OOS 폴드 평균
accuracy,0.3940
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0028
macro_f1,0.3760
balanced_accuracy,0.3802
mcc,0.0744
pr_auc_macro_ovr,0.3852
down_recall,0.3002
core_harmonic_mean,0.3513


재실행 명령: python scripts/run_stock_model_experiment.py
